In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import tensorflow as tf 
from tensorflow.keras import Sequential 
from tensorflow.keras.preprocessing.image import ImageDataGenerator 
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.layers import Dense, Flatten, Input, Dropout
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")

print("Path to dataset files:", path)

In [ ]:
import os

path = "data/tawsifurrahman/covid19-radiography-database/versions/5"
src = os.path.join(path, "COVID-19_Radiography_Dataset")
dst = "xray_data"

mapping = {"COVID": "covid-19", "Normal": "normal", "Viral Pneumonia": "pneumonia"}

for s, d in mapping.items():
    os.makedirs(os.path.join(dst, d), exist_ok=True)
    img_dir = os.path.join(src, s, "images")
    for f in os.listdir(img_dir):
        link = os.path.join(dst, d, f)
        if not os.path.exists(link):
            os.symlink(os.path.abspath(os.path.join(img_dir, f)), link)

print({c: len(os.listdir(os.path.join(dst, c))) for c in os.listdir(dst)})

In [ ]:
base = Xception(weights='imagenet', input_shape=(299,299,3), include_top=False)
for layers in base.layers:
    layers.trainable = False
base.summary()

In [ ]:
batch_size = 8 

train_datagen = ImageDataGenerator(rescale=1./255, shear_range=0.2, zoom_range=0.2,
                                   horizontal_flip=True, validation_split=0.25)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.25)

train = train_datagen.flow_from_directory("xray_data/", target_size=(299,299), batch_size=batch_size,
                                          class_mode='categorical', subset='training', seed=42)
val = val_datagen.flow_from_directory("xray_data/", target_size=(299,299), batch_size=batch_size,
                                      class_mode='categorical', subset='validation', shuffle=False)

class_names = list(train.class_indices.keys())
print(train.num_classes, train.class_indices, train.samples, val.samples)

In [ ]:
def plotImages(images_arr, labels):
    fig, axes = plt.subplots(12,4,figsize=(20,80))
    axes = axes.flatten()
    label = 0
    for img, ax in zip(images_arr, axes):
        ax.imshow(img)
        ax.set_title(class_names[np.argmax(labels[label])])
        label += 1
    plt.show()

In [ ]:
images = [train[34][0][0] for i in range(16)]
images = images + [train[5][0][0] for i in range(16)]
images = images + [train[0][0][0] for i in range(16)]

labels = []
labels = [train[34][1][0] for i in range(16)]
labels = labels + [train[5][1][0] for i in range(16)]
labels = labels + [train[0][1][0] for i in range(16)]

plotImages(images, labels)

In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling2D

model = Sequential()
model.add(Input(shape=(299,299,3)))
model.add(base)
model.add(GlobalAveragePooling2D())
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dense(3, activation='softmax'))
model.summary()

In [ ]:
print(tf.config.list_physical_devices("GPU"))

In [ ]:
from tensorflow.keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=["accuracy"])
model.fit(train, epochs=30, validation_data=val)
model.save('epoch_30.h5')